# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [10]:
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [24]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    
    B, C, H, W = x.shape
    if H % patch_size != 0:
        raise ValueError("Height not divisible by patch size")
    if W % patch_size != 0:
        raise ValueError("Width not divisible by patch size")
    
    pH, pW = H//patch_size, W//patch_size
    patches = x.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size) 
    # (B, C, sH, sW, p, p)
    patches = patches.contiguous().view(B, pH*pW, C*patch_size*patch_size)
    return patches

In [12]:
# Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        self.proj = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        return self.proj(x_patches)

class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, num_tokens, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos

In [13]:
# Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(), nn.Linear(d_ff, d_model), nn.Dropout())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        self.variant = variant
        self.w1 = nn.Linear(d_model, d_ff_gated)
        self.w2 = nn.Linear(d_model, d_ff_gated)
        self.out = nn.Linear(d_ff_gated, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = self.w1(x)
        b = self.w2(x)

        if self.variant == "geglu":
            h = F.gelu(a) * b
        elif self.variant == "swiglu":
            h = F.silu(a) * b
        else:
            raise ValueError(f"Invalid variant {self.variant}")
        
        h = self.dropout(h)
        return self.out(h)

In [26]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2= nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.mlp = mlp
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.ln1(x)
        attn_out, _ = self.attn(y, y, y)

        x = x + self.dropout(attn_out)
        y = self.ln2(x)
        x = x + self.dropout(self.mlp(y))
        return x

In [15]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        self.patch_embed = PatchEmbed(patch_dim=patch_dim, d_model=d_model)
        self.pos_embed = PositionalEmbedding(num_tokens=self.num_tokens, d_model=d_model)

        # implement a strategy to select the right mlp version for your experiment

        blocks = []
        for n in range(n_layers):

            if mlp_kind == "ffn":
                mlp = FeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
            else:
                mlp = GLUFeedForward(d_model=d_model, d_ff_gated=int((2/3) * d_ff), dropout=dropout, variant=mlp_kind)
            
            blocks.append(TransformerEncoderBlock(d_model=d_model, n_heads=n_heads, mlp=mlp, dropout=dropout))

        self.blocks = nn.ModuleList(blocks)
            
        # Add a head to project to the amount of output classes you have
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model,  10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = patchify(x, self.patch_size)

        x = self.patch_embed(x)
        x = self.pos_embed(x)

        for block in self.blocks:
            x = block(x)
        
        x = x.mean(dim=1)
        x = self.norm(x)
        
        logits = self.head(x)
        return logits

In [16]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [ ]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []

    for epoch in range(cfg.epochs):

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        # Evaluation loop. NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        print(f"[{mlp_kind}] epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")
    return {
        # Return your metrics that you think will support your claim for this experiment
        "kind": mlp_kind,
        "train_loss": train_losses,
        "test_acc": test_accs,
        "best_acc": max(test_accs),
        "final_acc": test_accs[-1]
    }

In [27]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=5, lr=3e-4, weight_decay=0.01, device="mps")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example. You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

runs = ["ffn", "geglu", "swiglu"] # Name your runs
results = []

for kind in runs:
    model = TinyViT(
        patch_size=patch_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        d_ff=d_ff,
        dropout=dropout,
        mlp_kind=kind,
    )
    n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
    )
    print(
    f"Parameters: {n_params:,}"
    )
    print(f"\nRun: {kind} | " )
    out = train_one_run(kind, model, train_loader, test_loader, cfg)
    results.append(out)

Parameters: 104,970

Run: ffn | 
[ffn] epoch 1/5 | test acc: 0.8278
[ffn] epoch 2/5 | test acc: 0.8909
[ffn] epoch 3/5 | test acc: 0.9108
[ffn] epoch 4/5 | test acc: 0.9255
[ffn] epoch 5/5 | test acc: 0.9338
Parameters: 104,882

Run: geglu | 
[geglu] epoch 1/5 | test acc: 0.8771
[geglu] epoch 2/5 | test acc: 0.9412
[geglu] epoch 3/5 | test acc: 0.9539
[geglu] epoch 4/5 | test acc: 0.9617
[geglu] epoch 5/5 | test acc: 0.9689
Parameters: 104,882

Run: swiglu | 
[swiglu] epoch 1/5 | test acc: 0.8505
[swiglu] epoch 2/5 | test acc: 0.9225
[swiglu] epoch 3/5 | test acc: 0.9358
[swiglu] epoch 4/5 | test acc: 0.9515
[swiglu] epoch 5/5 | test acc: 0.9608
